## Setup and Imports

In [85]:
import os
import numpy as np
import torch
from transformers import (
    AutoTokenizer, 
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification
)
from torch.utils.data import Dataset
import pandas as pd
from tqdm import tqdm

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print("Setup complete")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

Setup complete
PyTorch version: 2.11.0.dev20260204+cu128
CUDA available: True


###  Define label space (entity types + BIO tagging)
 definition of the 13 GutBrainIE entity categories and expansions into BIO tags:
 - "O" for tokens outside any entity
 - "B-<label>" for the first token of an entity mention
 - "I-<label>" for continuation tokens
 Then we build `label2id` / `id2label` mappings so the model can train and decode labels.

 Finally we set:
 - the pretrained backbone (BioBERT)
 - the output directory where the fine-tuned model will be saved.

In [86]:
# Define entity labels
ENTITY_LABELS = [
    "anatomical location",
    "animal",
    "bacteria",
    "biomedical technique",
    "chemical",
    "DDF",
    "dietary supplement",
    "drug",
    "food",
    "gene",
    "human",
    "microbiome",
    "statistical technique"
]

# Create BIO tags for each entity label
label_list = ['O']  # Outside
for entity_label in ENTITY_LABELS:
    label_list.append(f'B-{entity_label}')  # Beginning
    label_list.append(f'I-{entity_label}')  # Inside

label2id = {k: v for v, k in enumerate(label_list)}
id2label = {v: k for v, k in enumerate(label_list)}

print(f"Total labels: {len(label_list)}")
print(f"\nFirst 10 labels: {label_list[:10]}")

# Model configuration
model_name = "dmis-lab/biobert-v1.1"  # BioBERT for biomedical text
output_model_dir = "models/bert_biomedbert_ner_twopass_gold_silver"

print(f"\nModel: {model_name}")
print(f"Output directory: {output_model_dir}")

Total labels: 27

First 10 labels: ['O', 'B-anatomical location', 'I-anatomical location', 'B-animal', 'I-animal', 'B-bacteria', 'I-bacteria', 'B-biomedical technique', 'I-biomedical technique', 'B-chemical']

Model: dmis-lab/biobert-v1.1
Output directory: models/bert_biomedbert_ner_twopass_gold_silver


## Data Loading Functions
This section defines two helper functions:
 - `load_ner_data`: loads multiple JSON annotation files and merges them into a single dictionary keyed by PMID.
 - `prepare_documents_for_ner`: splits each article into two separate training examples:  one for the title and one for the abstract. This is important because entities are
  annotated with a `location` field (title/abstract) and the spans are relative to that text segment.

In [87]:
from pathlib import Path


def load_json(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

def load_ner_data_priority(train_paths_by_quality):
    """
    train_paths_by_quality: list of tuples (quality_name, path)
    Priority order is the given order (first wins).
    """
    merged = {}
    source = {}  # pmid -> quality

    for quality, path in train_paths_by_quality:
        if not path.exists():
            print(f"⚠️ Missing: {path}")
            continue

        data = load_json(path)
        print(f"Loaded {len(data)} docs from {path.name} ({quality})")

        for pmid, article in data.items():
            # first wins: gold > silver > bronze
            if pmid not in merged:
                merged[pmid] = article
                source[pmid] = quality

    print(f"✓ Merged unique PMIDs: {len(merged)} (priority kept: gold>silver>bronze)")
    return merged

def prepare_documents_for_ner(data):
    documents = []
    for pmid, article in data.items():
        meta = article.get("metadata", {})
        title_text = (meta.get("title") or "").strip()
        abstract_text = (meta.get("abstract") or "").strip()
        entities = article.get("entities", []) or []

        # title segment
        if title_text:
            title_entities = [e for e in entities if e.get("location") == "title"]
            documents.append({
                "pmid": str(pmid),
                "location": "title",
                "text": title_text,
                "entities": title_entities
            })

        # abstract segment
        if abstract_text:
            abstract_entities = [e for e in entities if e.get("location") == "abstract"]
            documents.append({
                "pmid": str(pmid),
                "location": "abstract",
                "text": abstract_text,
                "entities": abstract_entities
            })

    return documents
print("✓ Data loading functions defined")

✓ Data loading functions defined


## Load Training and Dev Data
loading of the training data (gold/platinum/silver) and development data from the provided dev split. Then it converts articles into per-segment examples (title + abstract), producing `train_documents` and `dev_documents`.

In [88]:
import json
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    """
    Risale le cartelle finché trova una directory che contiene 'data'.
    Funziona anche se il notebook parte da src/ner/...
    """
    start = start.resolve()
    for p in [start] + list(start.parents):
        if (p / "data").exists():
            return p
    raise FileNotFoundError(f"Non trovo la cartella 'data' risalendo da: {start}")

PROJECT_ROOT = find_repo_root(Path.cwd())
DATA_ROOT = PROJECT_ROOT / "data" / "GutBrainIE_Full_Collection_2026"
ANNOTATIONS_DIR = DATA_ROOT / "Annotations"

# --- Train files (2026) ---
train_files = [
    ANNOTATIONS_DIR / "Train" / "gold_quality"   / "json_format" / "train_gold.json",
    ANNOTATIONS_DIR / "Train" / "silver_quality" / "json_format" / "train_silver.json",
]

# opzionale: aggiungi anche silver_2025 se vuoi
train_files_optional = [
    ANNOTATIONS_DIR / "Train" / "silver_quality" / "json_format" / "train_silver_2025.json"
]

# --- Dev file ---
dev_file = ANNOTATIONS_DIR / "Dev" / "json_format" / "dev.json"

print("Train files:")
for p in train_files + train_files_optional:
    print(" -", p, "| exists:", p.exists())
print("Dev file:", dev_file, "| exists:", dev_file.exists())

Train files:
 - C:\Users\super\Documents\UniPd\ATA\GutBrainIE\data\GutBrainIE_Full_Collection_2026\Annotations\Train\gold_quality\json_format\train_gold.json | exists: True
 - C:\Users\super\Documents\UniPd\ATA\GutBrainIE\data\GutBrainIE_Full_Collection_2026\Annotations\Train\silver_quality\json_format\train_silver.json | exists: True
 - C:\Users\super\Documents\UniPd\ATA\GutBrainIE\data\GutBrainIE_Full_Collection_2026\Annotations\Train\silver_quality\json_format\train_silver_2025.json | exists: True
Dev file: C:\Users\super\Documents\UniPd\ATA\GutBrainIE\data\GutBrainIE_Full_Collection_2026\Annotations\Dev\json_format\dev.json | exists: True


In [89]:
# --- Train merge with priority ---
train_paths_by_quality = [
    ("gold",   train_files[0]),
    ("silver", train_files[1]),
]

train_data = load_ner_data_priority(train_paths_by_quality)
train_documents = prepare_documents_for_ner(train_data)
print("Total train segments (title+abstract):", len(train_documents))

# --- Optional: add silver_2025 by priority AFTER gold, BEFORE bronze (se vuoi) ---
# Se lo vuoi includere, fai un merge separato:
# train_paths_by_quality = [("gold", ...), ("silver_2025", ...), ("silver", ...), ("bronze", ...)]

# --- Dev ---
if not dev_file.exists():
    raise FileNotFoundError(f"Dev file not found: {dev_file}")

dev_data = load_json(dev_file)
dev_documents = prepare_documents_for_ner(dev_data)
print("Total dev segments (title+abstract):", len(dev_documents))

Loaded 639 docs from train_gold.json (gold)
Loaded 811 docs from train_silver.json (silver)
✓ Merged unique PMIDs: 1450 (priority kept: gold>silver>bronze)
Total train segments (title+abstract): 2900
Total dev segments (title+abstract): 160


In [90]:
# Show example document
example_doc = train_documents[10]
print(f"Example document:")
print(f"  PMID: {example_doc['pmid']}")
print(f"  Location: {example_doc['location']}")
print(f"  Text: {example_doc['text'][:200]}...")
print(f"  Number of entities: {len(example_doc['entities'])}")
print(f"\nFirst 3 entities:")
for entity in example_doc['entities'][:3]:
    print(f"    - '{entity['text_span']}' [{entity['label']}] @ {entity['start_idx']}-{entity['end_idx']}")

Example document:
  PMID: 35833267
  Location: title
  Text: MiR-483-3p improves learning and memory abilities via XPO1 in Alzheimer's disease....
  Number of entities: 3

First 3 entities:
    - 'MiR-483-3p' [chemical] @ 0-9
    - 'XPO1' [chemical] @ 54-57
    - 'Alzheimer's disease' [DDF] @ 62-80


## Initialize BERT Model and Tokenizer
This cell loads
- the BioBERT tokenizer
 - the BioBERT model with a token-classification head sized to our BIO label space

In [91]:
# Initialize tokenizer and model
print("Initializing BERT tokenizer and model...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(
    model_name, 
    num_labels=len(label_list), 
    id2label=id2label, 
    label2id=label2id
)

print(f"✓ Tokenizer loaded: {tokenizer.__class__.__name__}")
print(f"✓ Model loaded with {model.num_labels} labels")

# Test tokenization
sample_text = "The gut microbiome plays a role in Parkinson's disease."
tokens = tokenizer.tokenize(sample_text)
print(f"\nSample tokenization: {tokens}")

Initializing BERT tokenizer and model...


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 345.58it/s, Materializing param=bert.encoder.layer.11.output.dense.weight]              
BertForTokenClassification LOAD REPORT from: dmis-lab/biobert-v1.1
Key                 | Status     | 
--------------------+------------+-
pooler.dense.bias   | UNEXPECTED | 
pooler.dense.weight | UNEXPECTED | 
classifier.bias     | MISSING    | 
classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✓ Tokenizer loaded: BertTokenizer
✓ Model loaded with 27 labels

Sample tokenization: ['The', 'gut', 'micro', '##bio', '##me', 'plays', 'a', 'role', 'in', 'Parkinson', "'", 's', 'disease', '.']


## BIO Tag Generation for Training Data

The dataset provides entity spans as character offsets in the raw text segment.
 BERT training, however, needs one label per token. This function performs that alignment:

**Key design choices:**
 1) We use `return_offsets_mapping=True` to get (start_char, end_char) for each token.
 2) We initialize all labels to "O".
 3) We sort entities by (start position, longer span first) to make overlap handling deterministic:
    if two entities overlap, the longer one is applied first.
 4) We mark tokens as part of an entity if their offset range overlaps the entity character span.
 5) We prevent double-labeling with `labeled_positions` (first entity wins).

 Important detail for GutBrainIE:
 - `end_idx` in the dataset is inclusive, while Hugging Face offsets are exclusive.
   We convert to exclusive end by using `end_idx + 1` before overlap checks.

In [92]:
def align_labels_with_tokens(text, entities, tokenizer, label2id, max_length=512):
    encoding = tokenizer(
        text,
        return_offsets_mapping=True,
        return_special_tokens_mask=True,
        add_special_tokens=True,
        truncation=True,
        max_length=max_length,
    )

    input_ids = encoding["input_ids"]
    attention_mask = encoding["attention_mask"]
    offset_mapping = encoding["offset_mapping"]          # (start,end) end exclusive
    special_mask = encoding["special_tokens_mask"]       # 1 if special token

    tokens = tokenizer.convert_ids_to_tokens(input_ids)
    labels = ["O"] * len(input_ids)

    sorted_entities = sorted(
        entities,
        key=lambda e: (int(e["start_idx"]), -(int(e["end_idx"]) - int(e["start_idx"]))),
    )

    labeled_positions = set()

    for ent in sorted_entities:
        ent_start = int(ent["start_idx"])
        ent_end_excl = int(ent["end_idx"]) + 1  # dataset end inclusive -> exclusive
        ent_label = str(ent["label"])

        ent_token_start = None
        ent_token_end = None

        for idx, ((tok_start, tok_end), is_special) in enumerate(zip(offset_mapping, special_mask)):
            if is_special == 1:
                continue
            tok_start = int(tok_start); tok_end = int(tok_end)
            if tok_end <= tok_start:
                continue

            if tok_start < ent_end_excl and tok_end > ent_start:
                if ent_token_start is None:
                    ent_token_start = idx
                ent_token_end = idx

        if ent_token_start is not None and ent_token_end is not None:
            for i in range(ent_token_start, ent_token_end + 1):
                if i in labeled_positions:
                    continue
                tag = f"B-{ent_label}" if i == ent_token_start else f"I-{ent_label}"
                if tag in label2id:
                    labels[i] = tag
                    labeled_positions.add(i)

    label_ids = [label2id.get(tag, label2id["O"]) for tag in labels]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": label_ids,
        "tokens": tokens,
    }

## Process Training and Dev Data with BIO Tags
This step applies the BIO alignment to every title/abstract segment

In [93]:
# Process training data
print("Processing training data...")
processed_train = []

for i, doc in enumerate(tqdm(train_documents, desc="Processing train")):
    processed = align_labels_with_tokens(
        doc['text'],
        doc['entities'],
        tokenizer,
        label2id
    )
    processed['pmid'] = doc['pmid']
    processed['location'] = doc['location']
    processed['text'] = doc['text']
    processed['entities'] = doc['entities']
    processed_train.append(processed)

print(f"✓ Training data processed: {len(processed_train)} segments")

Processing training data...


Processing train: 100%|██████████| 2900/2900 [00:08<00:00, 334.54it/s]

✓ Training data processed: 2900 segments


In [94]:
# Process dev data
print("Processing dev data...")
processed_dev = []

for i, doc in enumerate(tqdm(dev_documents, desc="Processing dev")):
    processed = align_labels_with_tokens(
        doc['text'],
        doc['entities'],
        tokenizer,
        label2id
    )
    processed['pmid'] = doc['pmid']
    processed['location'] = doc['location']
    processed['text'] = doc['text']
    processed['entities'] = doc['entities']
    processed_dev.append(processed)

print(f"✓ Dev data processed: {len(processed_dev)} segments")

Processing dev data...


Processing dev: 100%|██████████| 160/160 [00:00<00:00, 303.03it/s]

✓ Dev data processed: 160 segments


In [95]:
# Show example with BIO tags
example_idx = 10
example = processed_train[example_idx]

print(f"Example from training data:")
print(f"  Text: {example['text'][:150]}...")
print(f"  Entities: {len(example['entities'])}")
print(f"\nToken-Label pairs (first 30):")

token_label_pairs = []
for token, label_id in zip(example['tokens'][:30], example['labels'][:30]):
    label = id2label[label_id]
    token_label_pairs.append((token, label))

df = pd.DataFrame(token_label_pairs, columns=['Token', 'Label'])
print(df.to_string(index=False))

Example from training data:
  Text: MiR-483-3p improves learning and memory abilities via XPO1 in Alzheimer's disease....
  Entities: 3

Token-Label pairs (first 30):
    Token      Label
    [CLS]          O
       Mi B-chemical
      ##R I-chemical
        - I-chemical
       48 I-chemical
      ##3 I-chemical
        - I-chemical
        3 I-chemical
      ##p I-chemical
  improve          O
      ##s          O
 learning          O
      and          O
   memory          O
abilities          O
      via          O
        X B-chemical
     ##PO I-chemical
      ##1 I-chemical
       in          O
Alzheimer      B-DDF
        '      I-DDF
        s      I-DDF
  disease      I-DDF
        .          O
    [SEP]          O


## Prepare Dataset for BERT Training

In [96]:
class NERDataset(Dataset):
    def __init__(self, processed_data):
        self.data = processed_data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            "input_ids": torch.tensor(item["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(item["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(item["labels"], dtype=torch.long),
        }

print("✓ Custom dataset class defined")

✓ Custom dataset class defined


In [97]:
# Create datasets
print("Creating training datasets...")

train_dataset = NERDataset(processed_train)
dev_dataset = NERDataset(processed_dev)

print(f"✓ Training dataset: {len(train_dataset)} examples")
print(f"✓ Dev dataset: {len(dev_dataset)} examples")

Creating training datasets...
✓ Training dataset: 2900 examples
✓ Dev dataset: 160 examples


## Configure Training Arguments

 Token classification requires padding sequences in a batch to the same length.
 The DataCollator:
 - pads input_ids/attention_mask
 - pads labels consistently (and uses -100 for ignored positions if configured by Trainer)
 This is required for correct batching with variable-length sequences.

In [98]:
# Setup data collator for token classification
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer, padding=True, return_tensors="pt")
print("✓ Data collator initialized")

✓ Data collator initialized


### Define Evaluation metric (seqeval over BIO tags)
 This cell defines a `compute_metrics_seqeval` function for Hugging Face Trainer.
 It converts model outputs (logits) into predicted BIO tags and compares them to gold BIO tags using seqeval.

**Implementation details:**
 - `argmax` selects the most likely tag per token.
 - tokens with label -100 are skipped (ignored padding/special positions).
 - seqeval computes precision/recall/F1 at the entity level from BIO sequences.

In [99]:
import torch
from seqeval.metrics import precision_score, recall_score, f1_score

def compute_metrics_seqeval(p):
    logits, labels = p
    preds = np.argmax(logits, axis=-1)

    true_labels = []
    true_preds = []

    for pred_seq, label_seq in zip(preds, labels):
        seq_true = []
        seq_pred = []
        for p_id, l_id in zip(pred_seq, label_seq):
            if l_id == -100:
                continue
            seq_true.append(id2label[int(l_id)])
            seq_pred.append(id2label[int(p_id)])
        true_labels.append(seq_true)
        true_preds.append(seq_pred)

    return {
        "precision": precision_score(true_labels, true_preds),
        "recall": recall_score(true_labels, true_preds),
        "f1": f1_score(true_labels, true_preds),
    }


##  Training hyperparameters (stability + stronger convergence)
 This cell sets the key training choices:
 - **learning_rate = 3e-5** with **warmup_ratio=0.1** for stability
 - **gradient_accumulation_steps=2** to increase effective batch size without extra GPU memory
- **num_train_epochs=5** to allow the NER head to converge better than short runs
 - best model selection based on seqeval F1
 - fp16 enabled if CUDA is available

In [100]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=output_model_dir,

    # Core optimization
    learning_rate=3e-5,                 # often better than 2e-5 for BioBERT NER
    lr_scheduler_type="linear",
    warmup_ratio=0.1,                   # critical for stability with higher LR
    weight_decay=0.01,

    # Batch/effective batch
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,      # effective batch = 16 (usually helps)

    # Training length
    num_train_epochs=5,                 # 3 is often too short for NER

    # Evaluation / checkpointing
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    label_smoothing_factor=0.0,  # keep 0 for token classification; don't smooth rare labels away
    # If you have compute_metrics with seqeval later, use f1
    metric_for_best_model="f1", #CHANGED
    greater_is_better=True,

    # Runtime / logging
    logging_steps=100,
    save_total_limit=2,
    seed=42,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

print("✓ Training configuration ready")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Learning rate: {training_args.learning_rate}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


✓ Training configuration ready
  Batch size: 8
  Epochs: 5
  Learning rate: 3e-05


## Train BERT Model


## Class-weighted loss (handle label imbalance)
 Many GutBrainIE labels are rare (e.g., some entity types appear much less than "O").
 This cell:
 1) Counts token-level label frequencies in the training set.
 2) Builds class weights using inverse-frequency^power (power=0.5 => sqrt inverse frequency).
 3) Normalizes weights to mean=1 and clips them to avoid extreme gradients

Then it defines a custom Trainer that overrides `compute_loss` to use:
  CrossEntropyLoss(weight=class_weights, ignore_index=-100)
 This generally improves recall for under-represented labels without exploding training.

In [101]:
import torch
from collections import Counter
from transformers import Trainer

def compute_class_weights(processed_train, num_labels, ignore_index=-100, power=0.5):
    """
    Compute class weights from token label counts.
    power=0.5 -> sqrt inverse frequency (usually stable).
    """
    counts = Counter()
    for ex in processed_train:
        for y in ex["labels"]:
            if y == ignore_index:
                continue
            counts[int(y)] += 1

    # build weights: w_c = (1 / freq_c)^power
    freqs = np.zeros(num_labels, dtype=np.float64)
    for c in range(num_labels):
        freqs[c] = counts.get(c, 0)

    # avoid div-by-zero for unseen classes (shouldn't happen, but safe)
    freqs[freqs == 0] = 1.0

    weights = (1.0 / freqs) ** power

    # normalize weights to mean=1 (keeps loss scale reasonable)
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float)

class WeightedLossTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**{k: v for k, v in inputs.items() if k != "labels"})
        logits = outputs.logits  # (B, T, C)

        # flatten
        loss_fct = torch.nn.CrossEntropyLoss(
            weight=self.class_weights.to(logits.device) if self.class_weights is not None else None,
            ignore_index=-100
        )
        loss = loss_fct(logits.view(-1, logits.size(-1)), labels.view(-1))

        return (loss, outputs) if return_outputs else loss


# --- compute weights and inspect FOOD-related weights ---
class_weights = compute_class_weights(processed_train, num_labels=len(label_list), power=0.5)
class_weights = torch.clamp(class_weights, min=0.5, max=5.0) #clipping added

print("Weight(B-food) =", float(class_weights[label2id["B-food"]]))
print("Weight(I-food) =", float(class_weights[label2id["I-food"]]))
print("Weight(O)      =", float(class_weights[label2id["O"]]))
pairs = [(id2label[i], float(class_weights[i])) for i in range(len(label_list))]
pairs_sorted = sorted(pairs, key=lambda x: x[1], reverse=True)
print("Top 10 highest weights:")
for lab, w in pairs_sorted[:10]:
    print(f"{lab:30s} {w:.3f}")



Weight(B-food) = 2.213557004928589
Weight(I-food) = 1.414275050163269
Weight(O)      = 0.5
Top 10 highest weights:
B-gene                         2.223
B-food                         2.214
B-statistical technique        2.046
B-drug                         1.835
B-dietary supplement           1.460
I-food                         1.414
B-animal                       1.267
B-anatomical location          1.132
B-biomedical technique         1.107
I-gene                         1.077


### Initialize Trainer (training loop + evaluation)

In [102]:
trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics_seqeval,
    class_weights=class_weights,
)


print("✓ Trainer initialized")
print(f"  Training samples: {len(train_dataset)}")
print(f"  Evaluation samples: {len(dev_dataset)}")

✓ Trainer initialized
  Training samples: 2900
  Evaluation samples: 160


### Train the model

In [103]:

print("="*60)
print("Starting model training...")
print("="*60)

import time
training_start_time = time.time()

train_result = trainer.train()

training_duration = time.time() - training_start_time

print("\n" + "="*60)
print("✓ TRAINING COMPLETED!")
print("="*60)
print(f"Training time: {training_duration/60:.2f} minutes")

Starting model training...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,3.572757,0.380261,0.602941,0.779661,0.680007
2,0.674576,0.320089,0.654959,0.832575,0.733163
3,0.514865,0.315210,0.689952,0.828855,0.753052
4,0.414372,0.330985,0.700798,0.834642,0.761887
5,0.375955,0.330358,0.714894,0.833402,0.769613


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.02it/s]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer


✓ TRAINING COMPLETED!
Training time: 3.95 minutes


## Save Trained Model

In [104]:
# Save the trained model
print("Saving trained model...")

os.makedirs(output_model_dir, exist_ok=True)
trainer.save_model(output_model_dir)
tokenizer.save_pretrained(output_model_dir)

print(f"✓ Model saved to: {output_model_dir}")

Saving trained model...


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.03it/s]

✓ Model saved to: models/bert_biomedbert_ner_twopass_gold_silver


## Load Model for Inference

In [105]:
# Load the trained model for inference
print("Loading trained model for inference...")

inference_model = AutoModelForTokenClassification.from_pretrained(output_model_dir)
inference_tokenizer = AutoTokenizer.from_pretrained(output_model_dir)
inference_model.eval()

if torch.cuda.is_available():
    inference_model = inference_model.cuda()

print(f"✓ Model loaded from: {output_model_dir}")

Loading trained model for inference...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 394.96it/s, Materializing param=classifier.weight]                                      


✓ Model loaded from: models/bert_biomedbert_ner_twopass_gold_silver


# INFERENCE

### Inference setup (load again + select device)

In [106]:
import numpy as np
import torch
from tqdm import tqdm
from transformers import AutoModelForTokenClassification, AutoTokenizer

# -------------------------
# 0) Load model + tokenizer
# -------------------------
print("Loading trained model for inference...")
inference_model = AutoModelForTokenClassification.from_pretrained(output_model_dir)
inference_tokenizer = AutoTokenizer.from_pretrained(output_model_dir)
inference_model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
inference_model.to(device)

print(f"✓ Model loaded from: {output_model_dir}")
print(f"✓ Device: {device}")




Loading trained model for inference...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 456.71it/s, Materializing param=classifier.weight]                                      


✓ Model loaded from: models/bert_biomedbert_ner_twopass_gold_silver
✓ Device: cuda



##  Two-pass threshold strategy (precision first, recall second)
 Instead of taking all decoded entities, we filter them by label-specific confidence thresholds.
 The idea:
 - Pass 1 uses strict thresholds to **keep only high-precision mentions**.
 - Pass 2 relaxes thresholds only for selected labels where recall is typically low.
#- Later **we merge the two passe**s while preventing overlaps with pass-1 outputs.

 This is a post-processing policy that can improve macro-F1 by balancing precision/recall per label.

In [107]:
# Pass 1: your best (high precision)
LABEL_THRESH_HIGH = {
  "DDF": 0.88,
  "bacteria": 0.84,
  "statistical technique": 0.91,
  "biomedical technique": 0.78,
  "gene": 0.68,
  "food": 0.60,
  "chemical": 0.72,
  "dietary supplement": 0.78,
  "drug": 0.80,
  "microbiome": 0.78,
  "anatomical location": 0.78,
  "human": 0.70,
  "animal": 0.70,
}

LABEL_THRESH_RECALL = {
    "food": 0.43,               # centro del range stabile 0.40–0.45
    "chemical": 0.65,           # best nella tua ricerca
    "bacteria": 0.80,           # best
    "dietary supplement": 0.72, # best (ma equivalente)
}
RECALL_LABELS = {"chemical", "food", "bacteria", "dietary supplement"}


DEFAULT_THRESH = 0.80

### Simple false-positive filters + label-specific postprocessing
This cell defines lightweight heuristics to remove obvious junk predictions:
- generic terms that are too unspecific (e.g., "microbes")
- markup fragments ("<...>")
- too short spans

It also adds a targeted postprocessing rule:
 - If something looks gene-like (e.g., IL-6, TNF-α, α-synuclein) but was predicted as "chemical",  remap it to "gene". This fixes a common confusion pattern that we observed.

In [108]:
import re
BAD_BACTERIA = {"bacteria", "micro", "microbes", "microorganisms", "genera", "taxa"}
BAD_CHEMICAL = {"metabolites", "neurotransmitters"}
BAD_DIETSUPP = {"nnss"}
BAD_MICROBIOME = {"micro", "microbiota", "gut"}
DIET_CONCEPT = {
    "diet", "ketogenic diet", "high-fat diet", "high fat diet",
    "high glycemic diet", "vegetarian diet", "balanced diet",
    "western diet", "mediterranean diet"
}
BAD_FOOD_EXACT = {
    "control", "ketogenic", "high-fat", "high fat", "high",
    "glycemic index", "lycemic index",
    "food", "ingested food"
}
def normalize_span(s: str) -> str:
    s = s.strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s

def apply_simple_filters(entities):
    cleaned = []
    for e in entities:
        s = normalize_span(e["text_span"])

        # drop obvious HTML/markup garbage
        if "<" in s or ">" in s:
            continue

        # drop empty/very short spans
        if len(s) <= 1:
            continue

        # bacteria generic junk
        if e["label"] == "bacteria" and s in BAD_BACTERIA:
            continue
        # dietary supplement junk
        if e["label"] == "dietary supplement" and s in BAD_DIETSUPP:
            continue
        if e["label"] == "chemical" and s in BAD_CHEMICAL:
            continue
        if e["label"] == "microbiome" and s in BAD_MICROBIOME:
            continue
        if e["label"] == "food":
            if s in DIET_CONCEPT or s.endswith(" diet"):
                continue
            if s in BAD_FOOD_EXACT:
                continue
        cleaned.append(e)
    return cleaned

# -------------------------
# 3) Gene vs Chemical postprocess
# -------------------------
GENE_LIKE = re.compile(
    r"^(il-\d+|tnf(-?α)?|ifn(-?γ)?|tgf(-?β\d*)?|snca|park7|dj-1|mapt|apoe\d*|hla-[a-z0-9\*\:]+)$",
    re.IGNORECASE,
)
CHEM_LIKE = re.compile(
    r"(aβ|amyloid|scfa|gaba|succinate|butyrate|propionate|acetate|\b[a-z]+ate\b|\b[a-z]+acid\b|\(\d+\-\d+\))",
    re.IGNORECASE,
)

def postprocess_gene_vs_chemical(entities):
    for e in entities:
        s = normalize_span(e["text_span"])
        if e["label"] == "chemical" and GENE_LIKE.match(s):
            e["label"] = "gene"
        elif e["label"] == "gene" and CHEM_LIKE.search(s):
            e["label"] = "chemical"
    return entities

FOOD_ANCHORS = {"kefir", "yogurt", "milk", "cheese", "cookie", "lentil", "lentils", "buckwheat", "wheat", "rice", "tea", "coffee"}
SUPP_HARD = {"capsule", "tablet", "extract", "powder"}
SUPP_SOFT = {"probiotic", "probiotics", "prebiotic", "prebiotics", "synbiotic", "synbiotics", "supplement"}

def postprocess_food_vs_supp(entities):
    for e in entities:
        s = normalize_span(e["text_span"])

        if any(w in s for w in FOOD_ANCHORS):
            # se è un alimento riconoscibile, trattalo come food
            if e["label"] in {"dietary supplement", "food"}:
                e["label"] = "food"
            continue

        if any(w in s for w in SUPP_HARD):
            if e["label"] in {"dietary supplement", "food"}:
                e["label"] = "dietary supplement"
            continue

        if e["label"] == "food" and any(w in s for w in SUPP_SOFT):
            e["label"] = "dietary supplement"
    return entities



### Core predictor: decode BIO + compute entity confidence score
This is the main inference function that:
1) tokenizes text with offsets
2) runs the model to get logits -> softmax probabilities
3) converts per-token predictions to BIO labels
4) rebuilds entity spans by scanning tokens left-to-right

**Scoring:**
 - For each entity we compute a confidence score as the mean token probability
 over the entity span (B-tag prob for first token + I-tag prob for continuation tokens).

**BIO repair:**
- If we see I-X without an active entity, we start a new entity (treat as B-X).
- If we see I-X but we are currently inside Y, we close Y and start X.

 These rules make decoding more robust to occasional BIO inconsistencies

In [109]:
def predict_entities_with_scores(
    model,
    tokenizer,
    text: str,
    id2label: dict,
    label2id: dict,
    max_length: int = 512,
):
    """
    Returns list of entities with:
      start_idx (inclusive), end_idx (inclusive), label, text_span, score
    Score = mean token probability over the entity span.

    BIO repair:
      - I-X without an active entity => start new entity as X (treat as B-X)
      - I-X with different active label => close current and start new X
    """
    if not text:
        return []

    enc = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        return_offsets_mapping=True,
        max_length=max_length,
    )

    offsets = enc.pop("offset_mapping")[0].cpu().numpy()
    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        out = model(**enc)
        logits = out.logits[0]  # [T, C]
        probs = torch.softmax(logits, dim=-1)  # [T, C]
        pred_ids = torch.argmax(probs, dim=-1).cpu().numpy()
        probs_cpu = probs.cpu().numpy()

    labels = [id2label[int(i)] for i in pred_ids]

    entities = []
    current = None

    def _start_entity(ent_label: str, s: int, e: int, t_idx: int):
        # e is exclusive in offsets; store inclusive in entity
        prob_idx = label2id.get(f"B-{ent_label}", None)
        token_prob = float(probs_cpu[t_idx, prob_idx]) if prob_idx is not None else float(probs_cpu[t_idx].max())
        return {
            "start_idx": s,
            "end_idx": e - 1,
            "label": ent_label,
            "text_span": text[s:e],
            "_token_probs": [token_prob],
        }

    def _extend_entity(ent: dict, e: int, t_idx: int):
        ent_label = ent["label"]
        prob_idx = label2id.get(f"I-{ent_label}", None)
        token_prob = float(probs_cpu[t_idx, prob_idx]) if prob_idx is not None else float(probs_cpu[t_idx].max())
        ent["end_idx"] = e - 1
        ent["text_span"] = text[ent["start_idx"]:e]
        ent["_token_probs"].append(token_prob)

    for t_idx, (lab, (s, e)) in enumerate(zip(labels, offsets)):
        s = int(s); e = int(e)

        # special tokens
        if s == 0 and e == 0:
            continue
        if e <= s:
            continue

        if lab.startswith("B-"):
            if current is not None:
                entities.append(current)
            ent_label = lab[2:]
            current = _start_entity(ent_label, s, e, t_idx)

        elif lab.startswith("I-"):
            ent_label = lab[2:]

            # --- BIO REPAIR ---
            if current is None:
                # I-X without B => start new X
                current = _start_entity(ent_label, s, e, t_idx)
                continue

            if ent_label != current["label"]:
                # I-X but current is Y => close Y and start X
                entities.append(current)
                current = _start_entity(ent_label, s, e, t_idx)
                continue

            # normal extend
            _extend_entity(current, e, t_idx)

        else:
            if current is not None:
                entities.append(current)
                current = None

    if current is not None:
        entities.append(current)

    # add score
    for ent in entities:
        probs_list = ent.pop("_token_probs", [])
        ent["score"] = float(np.mean(probs_list)) if probs_list else 0.0

    return entities



###  Threshold filtering (label -> threshold map)
 This helper keeps an entity only if:
- its computed score >= the threshold configured for its label
-  Labels not explicitly listed use `DEFAULT_THRESH`

In [110]:
def passes_threshold(e, label_thresh):
    thr = label_thresh.get(e["label"], DEFAULT_THRESH)
    s = normalize_span(e["text_span"])

    # food: se c'è "diet" alza la soglia (evita FP tipo "ketogenic diet")
    if e["label"] == "food" and ("diet" in s):
        thr = max(thr, 0.70)

    return e.get("score", 0.0) >= thr


def filter_by_threshold_with_map(entities, label_thresh):
    return [e for e in entities if passes_threshold(e, label_thresh)]


### Merge policy (no overlap with pass-1)
We merge two entity lists (pass-1 and pass-2) with a conservative rule:
- Keep all pass-1 (high precision)
- Add pass-2 entities only for selected labels (`RECALL_LABELS`)
- Add only if their character span does not overlap any already kept entity in the same segment
This avoids creating duplicated/competing spans that often hurt precision.

In [111]:
def span_iou(a, b):
    inter = max(0, min(a["end_idx"], b["end_idx"]) - max(a["start_idx"], b["start_idx"]) + 1)
    if inter == 0:
        return 0.0
    la = a["end_idx"] - a["start_idx"] + 1
    lb = b["end_idx"] - b["start_idx"] + 1
    return inter / (la + lb - inter)


def any_overlap(ent, kept, iou_thr=0.5):
    for k in kept:
        if k["location"] != ent["location"]:
            continue
        if ent["label"] == "food":
            # blocca solo se è quasi identico (doppione vero)
            if span_iou(ent, k) >= 0.85 and ent["label"] == k["label"]:
                return True
            continue

        if span_iou(ent, k) >= iou_thr:
            return True
    return False

def merge_two_pass(ents_high, ents_rec, recall_labels):
    kept = list(ents_high)
    for e in ents_rec:
        if e["label"] not in recall_labels:
            continue
        if not any_overlap(e, kept):
            kept.append(e)
    return kept


### Two-pass segment predictor + dev inference loop
 This function wraps the full inference pipeline for one text segment:
 - decode entities with scores
 - pass 1: strict thresholds + filters + postprocessing
 - pass 2: relaxed thresholds (selected labels) + filters + postprocessing
 - merge with "no overlap with pass-1" policy
 - remove the score field so the output matches submission schema

 Then we run it across all dev segments and aggregate entities back by PMID.

In [112]:
TRIM_CHARS = " \t\n\r.,;:()[]{}<>\"'"

def trim_entity_span(e, text):
    s = int(e["start_idx"])
    end = int(e["end_idx"])

    # safe bounds
    s = max(0, min(s, len(text)))
    end = max(0, min(end, len(text)-1))

    # trim left
    while s <= end and text[s] in TRIM_CHARS:
        s += 1
    # trim right
    while end >= s and text[end] in TRIM_CHARS:
        end -= 1

    if s <= end:
        e["start_idx"] = s
        e["end_idx"] = end
        e["text_span"] = text[s:end+1]
    return e


In [113]:
def predict_segment_entities_two_pass(model, tokenizer, text, location):
    ents_raw = predict_entities_with_scores(
        model=model,
        tokenizer=tokenizer,
        text=text,
        id2label=id2label,
        label2id=label2id,
        max_length=512,
    )
    ents_raw = [trim_entity_span(e, text) for e in ents_raw]

    # pass 1 (high precision)
    ents_high = filter_by_threshold_with_map(ents_raw, LABEL_THRESH_HIGH)
    ents_high = apply_simple_filters(ents_high)
    ents_high = postprocess_gene_vs_chemical(ents_high)
    ents_high = postprocess_food_vs_supp(ents_high)


    for e in ents_high:
        e["location"] = location

    # pass 2 (recall)
    ents_rec = filter_by_threshold_with_map(ents_raw, LABEL_THRESH_RECALL)
    ents_rec = apply_simple_filters(ents_rec)
    ents_rec = postprocess_gene_vs_chemical(ents_rec)
    ents_rec  = postprocess_food_vs_supp(ents_rec)
    for e in ents_rec:
        e["location"] = location

    merged = merge_two_pass(ents_high, ents_rec, recall_labels=RECALL_LABELS)

    # Remove score for submission compatibility
    for e in merged:
        e.pop("score", None)

    return merged

# -------------------------
# 8) Predict on dev set
# -------------------------
print("Running inference on dev set (two-pass)...")

predictions_two_pass = {}

for doc in tqdm(dev_documents, desc="Predicting (two-pass)"):
    pmid = doc["pmid"]
    location = doc["location"]
    text = doc["text"]

    ents = predict_segment_entities_two_pass(inference_model, inference_tokenizer, text, location)

    predictions_two_pass.setdefault(pmid, {"entities": []})
    predictions_two_pass[pmid]["entities"].extend(ents)

print(f"✓ Inference completed: {len(predictions_two_pass)} documents")
total_entities = sum(len(p["entities"]) for p in predictions_two_pass.values())
print(f"  Total entities predicted: {total_entities}")


Running inference on dev set (two-pass)...


Predicting (two-pass): 100%|██████████| 160/160 [00:03<00:00, 47.15it/s]

✓ Inference completed: 80 documents
  Total entities predicted: 2205


## Save Predictions

In [114]:
from pathlib import Path

# trova root del repo (cartella che contiene 'src')
def find_repo_root(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / "src").exists():
            return p
    raise FileNotFoundError("Cannot find repo root (folder containing 'src')")

PROJECT_ROOT = find_repo_root(Path.cwd())

pred_dir = PROJECT_ROOT / "src" / "predictions"
pred_dir.mkdir(parents=True, exist_ok=True)

output_path = pred_dir / "bert_NER_twopass_gold_silver.json"

with output_path.open("w", encoding="utf-8") as f:
    json.dump(predictions_two_pass, f, ensure_ascii=False, indent=2)

print("Saved to:", output_path)

Saved to: C:\Users\super\Documents\UniPd\ATA\GutBrainIE\src\predictions\bert_NER_twopass_gold_silver.json


## Error inspection on predictions
 1) **Per-label** precision/recall/F1 using exact span match.
 2) Overlap-based **confusion matrix** (best IoU match) to see label confusions.
 3) **Boundary error report**: correct label but wrong offsets (plus "near misses" within ±k chars).
 4) Most frequent **false-positive** strings per label (helps refine filters).

These tools are meant for iterative improvement (thresholds, filters, span handling).

In [115]:
from collections import defaultdict
import pandas as pd
import re

# ----------------------------
# Helpers
# ----------------------------
def norm_span(s: str) -> str:
    """Normalize span text for pattern analysis (FP strings)."""
    s = s.strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s

def ent_key(ent):
    # inclusive end_idx per your format
    return (int(ent["start_idx"]), int(ent["end_idx"]), str(ent["location"]), str(ent["label"]))

def ent_key_no_label(ent):
    return (int(ent["start_idx"]), int(ent["end_idx"]), str(ent["location"]))

def as_span(ent):
    # return (start, end_inclusive)
    return (int(ent["start_idx"]), int(ent["end_idx"]))

def overlap_len(a_start, a_end, b_start, b_end):
    # inclusive ends
    left = max(a_start, b_start)
    right = min(a_end, b_end)
    return max(0, right - left + 1)

def iou(a_start, a_end, b_start, b_end):
    inter = overlap_len(a_start, a_end, b_start, b_end)
    if inter == 0:
        return 0.0
    a_len = a_end - a_start + 1
    b_len = b_end - b_start + 1
    union = a_len + b_len - inter
    return inter / union

def build_index(entities):
    """
    Build location-based index for quick overlap checks.
    entities: list of dicts each with start_idx/end_idx/location/label/text_span
    """
    idx = defaultdict(list)  # loc -> list[(start,end,ent)]
    for e in entities:
        s, eend = as_span(e)
        loc = str(e["location"])
        idx[loc].append((s, eend, e))
    # sort by start for mild speed-up
    for loc in idx:
        idx[loc].sort(key=lambda x: x[0])
    return idx

def best_overlap_match(gold_ent, pred_candidates, min_iou=0.1):
    """
    Return best predicted entity overlapping the gold one, by IoU (ties by overlap length).
    pred_candidates: list[(start,end,ent)]
    """
    gs, ge = as_span(gold_ent)
    best = None
    best_iou = 0.0
    best_ol = 0

    for ps, pe, pent in pred_candidates:
        ol = overlap_len(gs, ge, ps, pe)
        if ol == 0:
            continue
        score = iou(gs, ge, ps, pe)
        if score < min_iou:
            continue
        if (score > best_iou) or (score == best_iou and ol > best_ol):
            best = pent
            best_iou = score
            best_ol = ol

    return best, best_iou, best_ol


# ----------------------------
# Flatten gold + pred
# ----------------------------
def flatten_gold(dev_data):
    gold = defaultdict(list)  # pmid -> list[ent]
    for pmid, article in dev_data.items():
        for e in article["entities"]:
            gold[pmid].append({
                "start_idx": int(e["start_idx"]),
                "end_idx": int(e["end_idx"]),
                "location": str(e["location"]),
                "label": str(e["label"]),
                "text_span": str(e.get("text_span", "")),
            })
    return gold

def flatten_pred(predictions):
    pred = defaultdict(list)
    for pmid, obj in predictions.items():
        for e in obj.get("entities", []):
            pred[pmid].append({
                "start_idx": int(e["start_idx"]),
                "end_idx": int(e["end_idx"]),
                "location": str(e["location"]),
                "label": str(e["label"]),
                "text_span": str(e.get("text_span", "")),
            })
    return pred


gold_by_pmid = flatten_gold(dev_data)
pred_by_pmid = flatten_pred(predictions_two_pass)

ALL_LABELS = sorted(set(
    [e["label"] for pmid in gold_by_pmid for e in gold_by_pmid[pmid]] +
    [e["label"] for pmid in pred_by_pmid for e in pred_by_pmid[pmid]]
))


# ----------------------------
# (1) Per-label Precision/Recall/F1
# ----------------------------
def per_label_prf(gold_by_pmid, pred_by_pmid, labels):
    gold_sets = {lab: set() for lab in labels}
    pred_sets = {lab: set() for lab in labels}

    for pmid, gold_ents in gold_by_pmid.items():
        for e in gold_ents:
            k = (pmid,) + ent_key(e)  # include pmid
            gold_sets[e["label"]].add(k)

    for pmid, pred_ents in pred_by_pmid.items():
        for e in pred_ents:
            k = (pmid,) + ent_key(e)
            pred_sets[e["label"]].add(k)

    rows = []
    for lab in labels:
        g = gold_sets[lab]
        p = pred_sets[lab]
        tp = len(g & p)
        fp = len(p - g)
        fn = len(g - p)

        prec = tp / (tp + fp + 1e-12)
        rec = tp / (tp + fn + 1e-12)
        f1 = 2 * prec * rec / (prec + rec + 1e-12)

        rows.append({
            "label": lab,
            "gold": len(g),
            "pred": len(p),
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "precision": prec,
            "recall": rec,
            "f1": f1,
        })

    df = pd.DataFrame(rows).sort_values("f1", ascending=False).reset_index(drop=True)
    return df

df_prf = per_label_prf(gold_by_pmid, pred_by_pmid, ALL_LABELS)
print("\n=== Per-label Precision / Recall / F1 ===")
print(df_prf.to_string(index=False, float_format=lambda x: f"{x:.4f}"))


# ----------------------------
# (2) Confusion matrix (overlap-based)
# ----------------------------
def confusion_matrix_overlap(gold_by_pmid, pred_by_pmid, labels, min_iou=0.1):
    conf = pd.DataFrame(0, index=labels + ["<NONE>"], columns=labels + ["<NONE>"], dtype=int)

    for pmid in gold_by_pmid.keys():
        gold_ents = gold_by_pmid.get(pmid, [])
        pred_ents = pred_by_pmid.get(pmid, [])
        pred_idx = build_index(pred_ents)

        used_pred = set()  # track exact pred entities used in matches (by object id tuple)
        # Map gold -> best pred overlap
        for g in gold_ents:
            loc = g["location"]
            best, best_iou, _ = best_overlap_match(g, pred_idx.get(loc, []), min_iou=min_iou)
            g_lab = g["label"]

            if best is None:
                conf.loc[g_lab, "<NONE>"] += 1
            else:
                bkey = (best["start_idx"], best["end_idx"], best["location"], best["label"], best.get("text_span",""))
                used_pred.add(bkey)
                conf.loc[g_lab, best["label"]] += 1

        # Preds with no overlap-match to any gold count as <NONE> -> pred_label
        gold_idx = build_index(gold_ents)
        for p in pred_ents:
            pkey = (p["start_idx"], p["end_idx"], p["location"], p["label"], p.get("text_span",""))
            if pkey in used_pred:
                continue
            loc = p["location"]
            best_gold, _, _ = best_overlap_match(p, gold_idx.get(loc, []), min_iou=min_iou)
            if best_gold is None:
                conf.loc["<NONE>", p["label"]] += 1

    return conf

conf = confusion_matrix_overlap(gold_by_pmid, pred_by_pmid, ALL_LABELS, min_iou=0.1)
print("\n=== Confusion Matrix (rows=gold, cols=pred, overlap-based) ===")
# show top-left slice if huge
print(conf.to_string())


# ----------------------------
# (3) Boundary error rate (same label, overlap but offsets differ)
#     + "near miss" within +/- k chars
# ----------------------------
def boundary_report(gold_by_pmid, pred_by_pmid, labels, min_iou=0.1, near_k=3):
    # counts per label
    exact_tp = Counter()
    boundary_mismatch = Counter()
    near_miss = Counter()

    for pmid in gold_by_pmid.keys():
        gold_ents = gold_by_pmid.get(pmid, [])
        pred_ents = pred_by_pmid.get(pmid, [])
        pred_idx = build_index(pred_ents)

        # exact label+offset TP set for fast check
        pred_exact = set((pmid,) + ent_key(e) for e in pred_ents)

        for g in gold_ents:
            lab = g["label"]
            gk = (pmid,) + ent_key(g)

            if gk in pred_exact:
                exact_tp[lab] += 1
                continue

            # find best overlapping pred
            loc = g["location"]
            best, best_iou, _ = best_overlap_match(g, pred_idx.get(loc, []), min_iou=min_iou)
            if best is None:
                continue

            # boundary mismatch: same label but not exact offsets
            if best["label"] == lab:
                boundary_mismatch[lab] += 1

                # near miss: offsets close (start/end within +/- near_k)
                if (abs(best["start_idx"] - g["start_idx"]) <= near_k) and (abs(best["end_idx"] - g["end_idx"]) <= near_k):
                    near_miss[lab] += 1

    rows = []
    for lab in labels:
        tp = exact_tp[lab]
        bm = boundary_mismatch[lab]
        nm = near_miss[lab]
        denom = tp + bm
        rate = bm / (denom + 1e-12)  # among correct-label matches, how often boundaries differ
        rows.append({
            "label": lab,
            "exact_TP": tp,
            "boundary_mismatch_same_label": bm,
            "boundary_error_rate": rate,
            f"near_miss_within_±{near_k}": nm,
        })

    df = pd.DataFrame(rows).sort_values("boundary_error_rate", ascending=False).reset_index(drop=True)
    return df

df_boundary = boundary_report(gold_by_pmid, pred_by_pmid, ALL_LABELS, min_iou=0.1, near_k=3)
print("\n=== Boundary Errors (same label overlap but offsets differ) ===")
print(df_boundary.to_string(index=False, float_format=lambda x: f"{x:.4f}"))


# ----------------------------
# (4) FP patterns: top false-positive strings per label
# ----------------------------
def fp_patterns(gold_by_pmid, pred_by_pmid, top_n=15):
    # gold exact set per pmid for TP check
    gold_exact = set()
    for pmid, gold_ents in gold_by_pmid.items():
        for e in gold_ents:
            gold_exact.add((pmid,) + ent_key(e))

    fp_by_label = defaultdict(Counter)

    for pmid, pred_ents in pred_by_pmid.items():
        for e in pred_ents:
            pk = (pmid,) + ent_key(e)
            if pk in gold_exact:
                continue  # true positive
            # false positive
            lab = e["label"]
            fp_by_label[lab][norm_span(e.get("text_span", ""))] += 1

    # Pretty print
    for lab, counter in sorted(fp_by_label.items(), key=lambda x: sum(x[1].values()), reverse=True):
        print(f"\n=== Top FP strings for label: {lab} (total FP={sum(counter.values())}) ===")
        for span, c in counter.most_common(top_n):
            if span == "":
                span = "<EMPTY>"
            print(f"{c:>4}  {span}")

fp_patterns(gold_by_pmid, pred_by_pmid, top_n=15)



=== Per-label Precision / Recall / F1 ===
                label  gold  pred  tp  fp  fn  precision  recall     f1
           microbiome   231   211 204   7  27     0.9668  0.8831 0.9231
                human   192   183 172  11  20     0.9399  0.8958 0.9173
                  DDF   793   697 650  47 143     0.9326  0.8197 0.8725
               animal   152   139 124  15  28     0.8921  0.8158 0.8522
                 drug    75    71  62   9  13     0.8732  0.8267 0.8493
  anatomical location   169   159 126  33  43     0.7925  0.7456 0.7683
             bacteria   183   149 119  30  64     0.7987  0.6503 0.7169
statistical technique    35    33  24   9  11     0.7273  0.6857 0.7059
             chemical   366   292 229  63 137     0.7842  0.6257 0.6960
   dietary supplement    64    60  43  17  21     0.7167  0.6719 0.6935
 biomedical technique   139   111  85  26  54     0.7658  0.6115 0.6800
                 gene    63    71  42  29  21     0.5915  0.6667 0.6269
                 food

## EVALUATION
Official-style evaluation (macro + micro)

In [116]:
# Load evaluation functions from evaluate.py concepts
def remove_duplicated_entities(predictions):
    """Remove duplicated entities from predictions."""
    removed_count = 0
    for pmid in list(predictions.keys()):
        seen = set()
        deduped = []
        for ent in predictions[pmid]["entities"]:
            #key = (ent["start_idx"], ent["end_idx"], ent["location"])
            key = (ent["start_idx"], ent["end_idx"], ent["location"], ent["label"])

            if key not in seen:
                seen.add(key)
                deduped.append(ent)
            else:
                removed_count += 1
        predictions[pmid]["entities"] = deduped
    
    if removed_count > 0:
        print(f"Removed {removed_count} duplicated entities from predictions")

def remove_overlapping_entities_eval(predictions):
    """Remove overlapping entities, keeping longest spans."""
    removed_count = 0

    for pmid in list(predictions.keys()):
        original_len = len(predictions[pmid]['entities'])
        
        groups = {'title': [], 'abstract': []}
        for ent in predictions[pmid]['entities']:
            loc = ent["location"]
            groups[loc].append(ent)

        keepers = set()
        for loc in groups:
            group = groups[loc]
            group = sorted(group, key=lambda e: e["start_idx"])

            clusters = []
            cluster = []
            current_end = None

            for ent in group:
                if not cluster:
                    cluster = [ent]
                    current_end = ent["end_idx"]
                else:
                    if ent["start_idx"] < current_end:
                        cluster.append(ent)
                        if ent["end_idx"] > current_end:
                            current_end = ent["end_idx"]
                    else:
                        clusters.append(cluster)
                        cluster = [ent]
                        current_end = ent["end_idx"]
            if cluster:
                clusters.append(cluster)

            for clust in clusters:
                longest = clust[0]
                max_len = longest["end_idx"] - longest["start_idx"]
                for ent in clust[1:]:
                    length = ent["end_idx"] - ent["start_idx"]
                    if length > max_len:
                        longest = ent
                        max_len = length
                keepers.add((longest["start_idx"],
                             longest["end_idx"],
                             longest["location"]))

        deduped = []
        for ent in predictions[pmid]['entities']:
            key = (ent["start_idx"], ent["end_idx"], ent["location"])
            if key in keepers:
                deduped.append(ent)
                keepers.remove(key)

        predictions[pmid]["entities"] = deduped
        removed_count += (original_len - len(deduped))

    if removed_count > 0:
        print(f"Removed {removed_count} overlapping entities")

print("✓ Evaluation helper functions defined")

✓ Evaluation helper functions defined


In [117]:
def evaluate_ner(predictions, ground_truth):
    """Evaluate NER predictions against ground truth."""
    # Remove duplicated and overlapping entities
    remove_duplicated_entities(predictions)
    remove_overlapping_entities_eval(predictions)
    
    LEGAL_ENTITY_LABELS = [
        "anatomical location", "animal", "bacteria", "biomedical technique",
        "chemical", "DDF", "dietary supplement", "drug", "food", "gene",
        "human", "microbiome", "statistical technique"
    ]
    
    ground_truth_NER = dict()
    count_annotated_entities_per_label = {}
    
    for pmid, article in ground_truth.items():
        if pmid not in ground_truth_NER:
            ground_truth_NER[pmid] = []
        for entity in article['entities']:
            start_idx = int(entity["start_idx"])
            end_idx = int(entity["end_idx"])
            location = str(entity["location"])
            text_span = str(entity["text_span"])
            label = str(entity["label"]) 
            
            entry = (start_idx, end_idx, location, text_span, label)
            ground_truth_NER[pmid].append(entry)
            
            if label not in count_annotated_entities_per_label:
                count_annotated_entities_per_label[label] = 0
            count_annotated_entities_per_label[label] += 1

    count_predicted_entities_per_label = {label: 0 for label in list(count_annotated_entities_per_label.keys())}
    count_true_positives_per_label = {label: 0 for label in list(count_annotated_entities_per_label.keys())}

    for pmid in predictions.keys():
        entities = predictions[pmid]['entities']
        
        for entity in entities:
            start_idx = int(entity["start_idx"])
            end_idx = int(entity["end_idx"])
            location = str(entity["location"])
            text_span = str(entity["text_span"])
            label = str(entity["label"]) 
            
            if label not in LEGAL_ENTITY_LABELS:
                continue

            if label in count_predicted_entities_per_label:
                count_predicted_entities_per_label[label] += 1

            entry = (start_idx, end_idx, location, text_span, label)
            if pmid in ground_truth_NER and entry in ground_truth_NER[pmid]:
                count_true_positives_per_label[label] += 1

    count_annotated_entities = sum(count_annotated_entities_per_label.values())
    count_predicted_entities = sum(count_predicted_entities_per_label.values())
    count_true_positives = sum(count_true_positives_per_label.values())

    micro_precision = count_true_positives / (count_predicted_entities + 1e-10)
    micro_recall = count_true_positives / (count_annotated_entities + 1e-10)
    micro_f1 = 2 * ((micro_precision * micro_recall) / (micro_precision + micro_recall + 1e-10))

    precision, recall, f1 = 0, 0, 0
    n = len(count_annotated_entities_per_label)
    for label in count_annotated_entities_per_label.keys():
        current_precision = count_true_positives_per_label[label] / (count_predicted_entities_per_label[label] + 1e-10) 
        current_recall = count_true_positives_per_label[label] / (count_annotated_entities_per_label[label] + 1e-10) 
        
        precision += current_precision
        recall += current_recall
        f1 += 2 * ((current_precision * current_recall) / (current_precision + current_recall + 1e-10))
    
    precision = precision / n
    recall = recall / n
    f1 = f1 / n

    return precision, recall, f1, micro_precision, micro_recall, micro_f1


# Evaluate
precision, recall, f1, micro_precision, micro_recall, micro_f1 = evaluate_ner(predictions_two_pass, dev_data)

print("="*60)
print("BERT NER BASELINE RESULTS")
print("="*60)
print("\nMacro-averaged Metrics:")
print(f"  Macro-Precision: {precision:.4f}")
print(f"  Macro-Recall:    {recall:.4f}")
print(f"  Macro-F1 Score:  {f1:.4f}")

print("\nMicro-averaged Metrics:")
print(f"  Micro-Precision: {micro_precision:.4f}")
print(f"  Micro-Recall:    {micro_recall:.4f}")
print(f"  Micro-F1 Score:  {micro_f1:.4f}")
print("="*60)

BERT NER BASELINE RESULTS

Macro-averaged Metrics:
  Macro-Precision: 0.8214
  Macro-Recall:    0.7184
  Macro-F1 Score:  0.7610

Micro-averaged Metrics:
  Micro-Precision: 0.8644
  Micro-Recall:    0.7560
  Micro-F1 Score:  0.8066


## Analysis: Entity Distribution by Label
- how many gold mentions exist in dev
- how many mentions your system predicts

 This helps spot systematic under/over-prediction:
 - predicted << gold => recall bottleneck for that label
- predicted >> gold => precision bottleneck for that label

In [118]:
from collections import Counter

# Count entities by label in predictions
pred_label_counts = Counter()
for pmid, pred in predictions_two_pass.items():
    for entity in pred['entities']:
        pred_label_counts[entity['label']] += 1

# Count entities by label in gold standard
gold_label_counts = Counter()
for pmid, article in dev_data.items():
    for entity in article['entities']:
        gold_label_counts[entity['label']] += 1

print("Entity Distribution by Label:")
print("="*60)
print(f"{'Label':<25} {'Gold':<10} {'Predicted':<10}")
print("-"*60)

all_labels = set(gold_label_counts.keys()) | set(pred_label_counts.keys())
for label in sorted(all_labels):
    print(f"{label:<25} {gold_label_counts[label]:<10} {pred_label_counts[label]:<10}")

print("-"*60)
print(f"{'TOTAL':<25} {sum(gold_label_counts.values()):<10} {sum(pred_label_counts.values()):<10}")

Entity Distribution by Label:
Label                     Gold       Predicted 
------------------------------------------------------------
DDF                       793        697       
anatomical location       169        159       
animal                    152        139       
bacteria                  183        149       
biomedical technique      139        111       
chemical                  366        292       
dietary supplement        64         60        
drug                      75         71        
food                      59         29        
gene                      63         71        
human                     192        183       
microbiome                231        211       
statistical technique     35         33        
------------------------------------------------------------
TOTAL                     2521       2205      
